# Menyiapkan Dataset

In [1]:
import random
import pandas as pd
import json

random.seed(701)
n = 400
gudang_list = ["Gudang Magelang", "Gudang Semarang", "Gudang Yogyakarta"]
kota_dirty = ["solo", "SOLO", "Purworejo", "purworejo", "Kebumen", "Temanggung", "wonosobo", "Wonosobo"]

kurir_ids = [f"KR{i:03d}" for i in range(1, 26)]
gudang_per_kurir = {kid: random.choice(gudang_list) for kid in kurir_ids}

rows = []
for i in range(n):
    kurir_id = random.choice(kurir_ids)
    rows.append({
        "pengiriman_id": f"PGR-{1000+i}",
        "kurir_id": kurir_id,
        "gudang_asal": gudang_per_kurir[kurir_id],
        "kota_tujuan": random.choice(kota_dirty),
        "berat_kg": round(random.uniform(0.5, 25), 1),
        "waktu_kirim_jam": random.randint(4, 48),
        "biaya_kirim": random.choice([15000,20000,25000,35000,50000]),
        "status_pengiriman": random.choice(
            ["Selesai","Selesai","Selesai","Dibatalkan"]),
    })
    
rows += [rows[3], rows[10], rows[50]]
for idx in [5, 15, 25, 100, 150]:
    rows[idx] = dict(rows[idx])
    rows[idx]["berat_kg"] = None
for idx in [7, 60, 120]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_kirim"] = None
for idx in [200, 250]:
    rows[idx] = dict(rows[idx])
    rows[idx]["berat_kg"] = -round(random.uniform(1,5),1)

pd.DataFrame(rows).to_csv("pengiriman.csv", index=False)

kurir_master = [
    {"kurir_id": kid, "nama_kurir": f"Kurir-{kid}",
    "gudang_asal": gudang_per_kurir[kid],
    "rating_kurir": round(random.uniform(3.0,5.0),1)}
    for kid in kurir_ids
]

with open("kurir.json", "w") as f:
    for k in kurir_master:
        f.write(json.dumps(k) + "\n")
        
print("Data berhasil dibuat: pengiriman.csv dan kurir.json")

Data berhasil dibuat: pengiriman.csv dan kurir.json


# Membuat SparkSession

In [2]:
import spark_session

app = spark_session.SparkApp("ResponsiUTS")
spark = app.getSession()

Memulai SparkSession untuk ResponsiUTS


26/10/01 08:46:29 WARN Utils: Your hostname, asfadani resolves to a loopback address: 127.0.1.1, but we couldn't find any external IP address!
26/10/01 08:46:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 08:46:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/01 08:46:30 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: e6:e9:4b:a9:29:1b:34:ff


SparkSession siap. Versi Spark: 3.5.9


In [3]:
from pyspark.sql.functions import sum as spark_sum, col, when, row_number, avg, count
from pyspark.sql.window import Window

# Ekstraksi Data melalui HDFS

**A. Unggah data ke HDFS**

In [5]:
!hdfs dfs -mkdir -p /user/asfadani/uts/raw/
!hdfs dfs -ls /user/asfadani/uts/

Found 1 items
drwxr-xr-x   - asfadani supergroup          0 2026-10-01 07:17 /user/asfadani/uts/raw


In [6]:
!hdfs dfs -put pengiriman.csv /user/asfadani/uts/raw/
!hdfs dfs -put kurir.json /user/asfadani/uts/raw/

In [7]:
!hdfs dfs -ls /user/asfadani/uts/raw/

Found 2 items
-rw-r--r--   1 asfadani supergroup       2666 2026-10-01 07:19 /user/asfadani/uts/raw/kurir.json
-rw-r--r--   1 asfadani supergroup      25914 2026-10-01 07:19 /user/asfadani/uts/raw/pengiriman.csv


**B. Baca pengiriman.csv dari HDFS (bukan dari disk lokal) menjadi Spark DataFrame.**

In [4]:
df_pengiriman = spark.read.csv("hdfs://localhost:9000/user/asfadani/uts/raw/pengiriman.csv", header=True, inferSchema=True)

**C. Baca kurir.json menjadi Spark DataFrame**

In [5]:
df_kurir = spark.read.json("hdfs://localhost:9000/user/asfadani/uts/raw/kurir.json")

**D. Tampilkan Print Schema nya dan jumlah baris untuk KEDUA DataFrame.**

In [6]:
df_pengiriman.printSchema()
print(f"Jumalah baris: {df_pengiriman.count()}")

df_kurir.printSchema()
print(f"Jumalah baris: {df_kurir.count()}")

root
 |-- pengiriman_id: string (nullable = true)
 |-- kurir_id: string (nullable = true)
 |-- gudang_asal: string (nullable = true)
 |-- kota_tujuan: string (nullable = true)
 |-- berat_kg: double (nullable = true)
 |-- waktu_kirim_jam: integer (nullable = true)
 |-- biaya_kirim: double (nullable = true)
 |-- status_pengiriman: string (nullable = true)

Jumalah baris: 403
root
 |-- gudang_asal: string (nullable = true)
 |-- kurir_id: string (nullable = true)
 |-- nama_kurir: string (nullable = true)
 |-- rating_kurir: double (nullable = true)

Jumalah baris: 25


# Eksplorasi dan Pembersihan Data

Data pengiriman.csv mengandung BEBERAPA masalah kualitas data yang lazim ditemukan pada data dunia
nyata. Lakukan eksplorasi untuk menemukan seluruh masalah tersebut, lalu tangani masing-masing.

**A. Identifikasi dan tampilkan bukti dari MINIMAL TIGA jenis masalah kualitas data yang berbeda pada
pengiriman.csv.**

In [7]:
df_pengiriman.show()
print(f"Jumalah baris: {df_pengiriman.count()}")

+-------------+--------+-----------------+-----------+--------+---------------+-----------+-----------------+
|pengiriman_id|kurir_id|      gudang_asal|kota_tujuan|berat_kg|waktu_kirim_jam|biaya_kirim|status_pengiriman|
+-------------+--------+-----------------+-----------+--------+---------------+-----------+-----------------+
|     PGR-1000|   KR011|  Gudang Semarang|    Kebumen|    23.4|             11|    25000.0|          Selesai|
|     PGR-1001|   KR018|  Gudang Magelang|       SOLO|    14.2|             31|    35000.0|          Selesai|
|     PGR-1002|   KR013|Gudang Yogyakarta|  purworejo|    10.6|              5|    15000.0|          Selesai|
|     PGR-1003|   KR024|  Gudang Semarang| Temanggung|    20.1|             19|    50000.0|       Dibatalkan|
|     PGR-1004|   KR015|  Gudang Semarang|       SOLO|     1.3|             47|    35000.0|          Selesai|
|     PGR-1005|   KR014|  Gudang Magelang|  Purworejo|    NULL|             34|    25000.0|          Selesai|
|     PGR-

>Dari menampilkan dataframe nya, dilihat bahwa masalah kulaitas data terdapat di kolom `berat_kg`(missing value), `biaya_kirim`(missing value), dan `kota_tujuan`(tidak seragam penulisannya)

**B. Tangani seluruh masalah yang ditemukan pada poin (a). Untuk SETIAP penanganan, tuliskan satu kalimat
singkat (markdown cell atau komentar kode) yang menjelaskan alasan pendekatan yang dipilih.**

Kolom`berat_kg` menangani missing valuenya dengan mengisikan rata rata beratnya (karena tidak mungkin berat barang 0kg)

In [8]:
# Menangani missing value berat_kg
berat = df_pengiriman.agg(
    avg(col("berat_kg"))
)

berat.show()

+------------------+
|     avg(berat_kg)|
+------------------+
|12.874623115577888|
+------------------+



In [9]:
import pyspark.pandas as ps

df1 = ps.DataFrame(df_pengiriman)
display(df1.head(10))

/home/asfadani/anaconda3/envs/bigdata/lib/python3.11/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


,pengiriman_id,kurir_id,gudang_asal,kota_tujuan,berat_kg,waktu_kirim_jam,biaya_kirim,status_pengiriman
0,PGR-1000,KR011,Gudang Semarang,Kebumen,23.4,11,25000.0,Selesai
1,PGR-1001,KR018,Gudang Magelang,SOLO,14.2,31,35000.0,Selesai
2,PGR-1002,KR013,Gudang Yogyakarta,purworejo,10.6,5,15000.0,Selesai
3,PGR-1003,KR024,Gudang Semarang,Temanggung,20.1,19,50000.0,Dibatalkan
4,PGR-1004,KR015,Gudang Semarang,SOLO,1.3,47,35000.0,Selesai
5,PGR-1005,KR014,Gudang Magelang,Purworejo,NaN,34,25000.0,Selesai
6,PGR-1006,KR017,Gudang Yogyakarta,Kebumen,22.7,26,50000.0,Dibatalkan
7,PGR-1007,KR023,Gudang Semarang,Kebumen,19.7,17,NaN,Dibatalkan
8,PGR-1008,KR006,Gudang Yogyakarta,wonosobo,18.5,35,15000.0,Selesai
9,PGR-1009,KR006,Gudang Yogyakarta,Kebumen,24.1,10,20000.0,Selesai


In [10]:
df1.isna().count()

pengiriman_id        403
kurir_id             403
gudang_asal          403
kota_tujuan          403
berat_kg             403
waktu_kirim_jam      403
biaya_kirim          403
status_pengiriman    403
dtype: int64

In [11]:
df1["berat_kg"] = df1["berat_kg"].fillna(12.8)

In [12]:
display(df1["berat_kg"].head(10))

0    23.4
1    14.2
2    10.6
3    20.1
4     1.3
5    12.8
6    22.7
7    19.7
8    18.5
9    24.1
Name: berat_kg, dtype: float64

kolom `biaya_kirim` missing valuenya akan diisi dengan 0 nya (bisa dianggap menggunakan gratis ongkir)

In [13]:
df1["biaya_kirim"] = df1["biaya_kirim"].fillna(0)

In [14]:
display(df1)

,pengiriman_id,kurir_id,gudang_asal,kota_tujuan,berat_kg,waktu_kirim_jam,biaya_kirim,status_pengiriman
0,PGR-1000,KR011,Gudang Semarang,Kebumen,23.4,11,25000.0,Selesai
1,PGR-1001,KR018,Gudang Magelang,SOLO,14.2,31,35000.0,Selesai
2,PGR-1002,KR013,Gudang Yogyakarta,purworejo,10.6,5,15000.0,Selesai
3,PGR-1003,KR024,Gudang Semarang,Temanggung,20.1,19,50000.0,Dibatalkan
4,PGR-1004,KR015,Gudang Semarang,SOLO,1.3,47,35000.0,Selesai
5,PGR-1005,KR014,Gudang Magelang,Purworejo,12.8,34,25000.0,Selesai
6,PGR-1006,KR017,Gudang Yogyakarta,Kebumen,22.7,26,50000.0,Dibatalkan
7,PGR-1007,KR023,Gudang Semarang,Kebumen,19.7,17,0.0,Dibatalkan
8,PGR-1008,KR006,Gudang Yogyakarta,wonosobo,18.5,35,15000.0,Selesai
9,PGR-1009,KR006,Gudang Yogyakarta,Kebumen,24.1,10,20000.0,Selesai


Menyeragamkan nama kota di kolom `kota_tujuan`

In [71]:
df_pengiriman.select("kota_tujuan").show()

+-----------+
|kota_tujuan|
+-----------+
|    Kebumen|
|       SOLO|
|  purworejo|
| Temanggung|
|       SOLO|
|  Purworejo|
|    Kebumen|
|    Kebumen|
|   wonosobo|
|    Kebumen|
|       solo|
|    Kebumen|
|   wonosobo|
|  purworejo|
|   wonosobo|
|  Purworejo|
|  Purworejo|
|   Wonosobo|
|    Kebumen|
|    Kebumen|
+-----------+
only showing top 20 rows



In [123]:
map_kota =  {
    "solo" : "Solo",
    "SOLO" : "Solo",
    "wonosobo" : "Wonosobo",
    "purworejo" : "Purworejo"
}

df1["kota_tujuan"] = df1["kota_tujuan"].map(map_kota)

ValueError: 'index' of dict type is not supported yet

# Transformasi Data dan Penggabungan

In [25]:
df_pengiriman_bersih = df1.to_spark()

In [57]:
df_pengiriman_bersih.show()

+-------------+--------+-----------------+-----------+--------+---------------+-----------+-----------------+
|pengiriman_id|kurir_id|      gudang_asal|kota_tujuan|berat_kg|waktu_kirim_jam|biaya_kirim|status_pengiriman|
+-------------+--------+-----------------+-----------+--------+---------------+-----------+-----------------+
|     PGR-1000|   KR011|  Gudang Semarang|    Kebumen|    23.4|             11|    25000.0|          Selesai|
|     PGR-1001|   KR018|  Gudang Magelang|       SOLO|    14.2|             31|    35000.0|          Selesai|
|     PGR-1002|   KR013|Gudang Yogyakarta|  purworejo|    10.6|              5|    15000.0|          Selesai|
|     PGR-1003|   KR024|  Gudang Semarang| Temanggung|    20.1|             19|    50000.0|       Dibatalkan|
|     PGR-1004|   KR015|  Gudang Semarang|       SOLO|     1.3|             47|    35000.0|          Selesai|
|     PGR-1005|   KR014|  Gudang Magelang|  Purworejo|    12.8|             34|    25000.0|          Selesai|
|     PGR-

In [71]:
df_gabungan = df_pengiriman_bersih.join(df_kurir, how="left", on="gudang_asal")

In [72]:
df_gabungan.show(5)

+---------------+-------------+--------+-----------+--------+---------------+-----------+-----------------+--------+-----------+------------+
|    gudang_asal|pengiriman_id|kurir_id|kota_tujuan|berat_kg|waktu_kirim_jam|biaya_kirim|status_pengiriman|kurir_id| nama_kurir|rating_kurir|
+---------------+-------------+--------+-----------+--------+---------------+-----------+-----------------+--------+-----------+------------+
|Gudang Semarang|     PGR-1000|   KR011|    Kebumen|    23.4|             11|    25000.0|          Selesai|   KR025|Kurir-KR025|         4.3|
|Gudang Semarang|     PGR-1000|   KR011|    Kebumen|    23.4|             11|    25000.0|          Selesai|   KR024|Kurir-KR024|         4.7|
|Gudang Semarang|     PGR-1000|   KR011|    Kebumen|    23.4|             11|    25000.0|          Selesai|   KR023|Kurir-KR023|         4.7|
|Gudang Semarang|     PGR-1000|   KR011|    Kebumen|    23.4|             11|    25000.0|          Selesai|   KR021|Kurir-KR021|         3.2|
|Gudan

In [73]:
df_gabungan = df_gabungan.withColumn("biaya_per_kg", col("biaya_kirim") / col("berat_kg"))

In [74]:
df_gabungan.select("biaya_per_kg").show(10)

+------------------+
|      biaya_per_kg|
+------------------+
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|1068.3760683760684|
|2464.7887323943664|
|2464.7887323943664|
+------------------+
only showing top 10 rows



# Analisis Window Functions

In [75]:
kurir = df_gabungan.groupBy("nama_kurir", "gudang_asal").agg(
    count(col("status_pengiriman")).alias("pengiriman_terbanyak"),
).orderBy(col("pengiriman_terbanyak").desc())

kurir.show(5)

+-----------+---------------+--------------------+
| nama_kurir|    gudang_asal|pengiriman_terbanyak|
+-----------+---------------+--------------------+
|Kurir-KR011|Gudang Semarang|                 137|
|Kurir-KR021|Gudang Semarang|                 137|
|Kurir-KR025|Gudang Semarang|                 137|
|Kurir-KR002|Gudang Semarang|                 137|
|Kurir-KR023|Gudang Semarang|                 137|
+-----------+---------------+--------------------+
only showing top 5 rows



In [77]:
window_kurir = Window.partitionBy("gudang_asal").orderBy(col("pengiriman_terbanyak").desc())
rank_kurir = kurir.withColumn("rank_pengiriman", row_number().over(window_kurir)).filter(col("rank_pengiriman") <=3 )

rank_kurir.show()

+-----------+-----------------+--------------------+---------------+
| nama_kurir|      gudang_asal|pengiriman_terbanyak|rank_pengiriman|
+-----------+-----------------+--------------------+---------------+
|Kurir-KR014|  Gudang Magelang|                 136|              1|
|Kurir-KR010|  Gudang Magelang|                 136|              2|
|Kurir-KR016|  Gudang Magelang|                 136|              3|
|Kurir-KR002|  Gudang Semarang|                 137|              1|
|Kurir-KR003|  Gudang Semarang|                 137|              2|
|Kurir-KR015|  Gudang Semarang|                 137|              3|
|Kurir-KR007|Gudang Yogyakarta|                 130|              1|
|Kurir-KR001|Gudang Yogyakarta|                 130|              2|
|Kurir-KR013|Gudang Yogyakarta|                 130|              3|
+-----------+-----------------+--------------------+---------------+



In [82]:
df2 = ps.DataFrame(df_gabungan)

AnalysisException: [AMBIGUOUS_REFERENCE] Reference `kurir_id` is ambiguous, could be: [`kurir_id`, `kurir_id`].